# ORIE 5380 · Homework 5

In [1]:
# Supplied setup: run this cell before the modeling cells.
%pip install -q gurobipy

import gurobipy as gp
from gurobipy import GRB

# A tiny test checks that the package can actually solve a model.
with gp.Model("setup_check") as check:
    check.Params.OutputFlag = 0
    check_x = check.addVar(ub=1, name="check_x")
    check.setObjective(check_x, GRB.MAXIMIZE)
    check.optimize()
    if check.Status == GRB.OPTIMAL:
        print("Ready for class: x =", check_x.X, "objective =", check.ObjVal)
    else:
        print("Setup check status:", check.Status)


Note: you may need to restart the kernel to use updated packages.
Restricted license - for non-production use only - expires 2027-11-29
Ready for class: x = 1.0 objective = 1.0


## 1. Covering problem
Model the covering problem in section 13.1 using Gurobi. Instead of adding the variables and constraints one by one, please add them using loops. Solve the integer program and report the optimal solution together with the optimal objective value.

$$
\begin{alignedat}{3}
\text{maximize}\quad & 1500y_1 + 2500y_3 + 3000y_4 + 2000y_5 \\[6pt]
\text{subject to}\quad & x_1 + x_2 + x_3    &\le &\;&2 \\
                    & y_1                   &\le &\;&x_1 + x_2 \\
                      & y_2                 &\le &&x_1 + x_3 \\
                      & y_3                 &\le &&x_2 \\
                      & y_4                 &\le &&x_1 \\
                      & y_5                 &\le &&x_2 + x_3.
\end{alignedat}
$$

### Create the model and its decision variables

I'm starting with individual constraints and then will build loop.

In [1]:
import gurobipy as gp
from gurobipy import GRB

#create the model
m = gp.Model("covering")

# Define each variable. We do not need bounds because the maximiazation 
# function takes care of it.
x1 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="x1")
x2 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="x2")
x3 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="x3")
y1 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="y1")
y2 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="y2")
y3 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="y3")
y4 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="y4")
y5 = m.addVar( lb=0, vtype=GRB.CONTINUOUS, name="y5")

# Add all the constraints
m.addConstr(x1 + x2 + x3    <= 2, name="c1")
m.addConstr(y1              <= x1 + x2, name="c2")
m.addConstr(y2              <= x1 + x3, name="c3")
m.addConstr(y3              <= x2, name="c4")
m.addConstr(y4              <= x1, name="c5")
m.addConstr(y5              <= x2 + x3, name="c6")

# Choose minimum or maximum
m.setObjective(1500*y1 + 2500*y3 + 3000*y4 + 2000*y5, GRB.MAXIMIZE)

# Optimize is the method that starts the solver, then run the cell.
m.optimize()

# Read solution values only after an optimal solve.
if m.Status == GRB.OPTIMAL:
    print("X1 =", x1.X)
    print("X2 =", x2.X)
    print("X3 =", x3.X)
    print("Y1 =", y1.X)
    print("Y2 =", y2.X)
    print("Y3 =", y3.X)
    print("Y4 =", y4.X)
    print("Y5 =", y5.X)
    print("Total population under coverage =", m.ObjVal)
else:
    print("Solve status:", m.Status)

# Check answer against original requirements
if m.Status == GRB.OPTIMAL:
    print("x1 + x2 + x3 =", x1.X + x2.X + x3.X)
    print("y1 = ", y1.X, " <= x1 + x2 =", x1.X + x2.X)
    print("y2 = ", y2.X, " <= x1 + x3 =", x1.X + x3.X)
    print("y3 = ", y3.X, " <= x2 =", x2.X)
    print("y4 = ", y4.X, " <= x1 =", x1.X)
    print("y5 = ", y5.X, " <= x2 + x3 =", x2.X + x3.X)

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G720)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 6 rows, 8 columns and 16 nonzeros (Max)
Model fingerprint: 0x45e76ece
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+03, 3e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 2e+00]

Presolve removed 6 rows and 8 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.2000000e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.200000000e+04
X1 = 0.0
X2 = 2.0
X3 = 0.0
Y1 = 2.0
Y2 = 0.0
Y3 = 2.0
Y4 = 0.0
Y5 = 2.0
Total population under coverage = 12000.0
x1 + x2 + x3 = 

Now for loops

In [3]:
import gurobipy as gp
from gurobipy import GRB

# Initialize the sets of x_i's and y_j's
I = [1, 2, 3]
J = [1, 2, 3, 4, 5]

# For each district y_j, the list of x_i's on the right-hand side of its constraint
# e.g. y1 <= x1 + x2  ->  cover[1] = [1, 2]
cover = {
    1: [1, 2],
    2: [1, 3],
    3: [2],
    4: [1],
    5: [2, 3],
}

# Create the model
m_loop = gp.Model("covering_loop")

# Add the x and y variables in loops
x = {}
for i in I:
    x[i] = m_loop.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"x{i}")
    
y = {}
for j in J:
    y[j] = m_loop.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"y{j}")

# First constraint
m_loop.addConstr(x[1] + x[2] + x[3] <= 2, name="c1")

# District constraints: y_j <= sum of the x_i's listed in cover[j]
for j in J:
    m_loop.addConstr(y[j] <= gp.quicksum(x[i] for i in cover[j]), name=f"c{j + 1}")

# Choose minimum or maximum
m_loop.setObjective(1500*y[1] + 2500*y[3] + 3000*y[4] + 2000*y[5], GRB.MAXIMIZE)

# Start the solver
m_loop.optimize()

# Read solution values only after an optimal solve.
if m_loop.Status == GRB.OPTIMAL:
    print("X1 =", x[1].X)
    print("X2 =", x[2].X)
    print("X3 =", x[3].X)
    print("Y1 =", y[1].X)
    print("Y2 =", y[2].X)
    print("Y3 =", y[3].X)
    print("Y4 =", y[4].X)
    print("Y5 =", y[5].X)
    print("Total population under coverage =", m_loop.ObjVal)
else:
    print("Solve status:", m_loop.Status)

# Check answer against original requirements
if m_loop.Status == GRB.OPTIMAL:
    print("x1 + x2 + x3 =", x[1].X + x[2].X + x[3].X)
    print("y1 = ", y[1].X, " <= x1 + x2 =", x[1].X + x[2].X)
    print("y2 = ", y[2].X, " <= x1 + x3 =", x[1].X + x[3].X)
    print("y3 = ", y[3].X, " <= x2 =", x[2].X)
    print("y4 = ", y[4].X, " <= x1 =", x[1].X)
    print("y5 = ", y[5].X, " <= x2 + x3 =", x[2].X + x[3].X)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G720)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 6 rows, 8 columns and 16 nonzeros (Max)
Model fingerprint: 0x45e76ece
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+03, 3e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 2e+00]

Presolve removed 6 rows and 8 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.2000000e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.200000000e+04
X1 = 0.0
X2 = 2.0
X3 = 0.0
Y1 = 2.0
Y2 = 0.0
Y3 = 2.0
Y4 = 0.0
Y5 = 2.0
Total population under coverage = 12000.0
x1 + x2 + x3 = 2.0
y1 =  2.0  <= x1 + x2 = 2.0
y2 =  0.0  <= x1 + x3 = 0.0
y3 =  2.0 